# Entrenamiento y comparación de modelos (v1.1.2)

Usa `ft_engineering()` para obtener los datos ya transformados y entrena varios clasificadores. Cada modelo se evalúa con un **umbral de decisión ajustado** según una **métrica de negocio**.

**Clase de interés:** `0` = NO paga a tiempo. Por eso precisión, recall y F1 se calculan con `pos_label=0`.

> Requiere `ft_engineering.py`, `cargar_datos.py` y `Base_de_datos.xlsx` (en la raíz del proyecto).
>
> Este notebook también se **importa** desde `model_evaluation.py`: las celdas de definición (funciones) se ejecutan al importar, y la celda final de ejecución solo corre cuando se abre el notebook directamente.

## 1. Librerías

In [9]:
import json
import os

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, roc_auc_score)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

from ft_engineering import ft_engineering  # datos limpios, divididos y transformados

# XGBoost es opcional: si no está instalado, se entrenan los demás modelos
try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None
    print("xgboost no está instalado: se entrenan los demás modelos.")

RANDOM_STATE = 1234  # semilla fija para resultados reproducibles

# Métricas de clasificación (sobre la clase 0 = no paga) y de negocio
METRICAS = ["accuracy", "precision", "recall", "f1_score", "roc_auc",
            "casosNoPagoAtiempo"]
METRICAS_NEGOCIO = ["valor_negocio", "pct_valor_capturado"]

## 2. Métrica de negocio personalizada

**Decisión que apoya el modelo:** si predice "no paga", el crédito se **rechaza**; si predice "paga", se aprueba. La referencia es aprobar a todos (valor 0).

| Resultado | Qué pasa | Efecto sobre el valor |
|---|---|---|
| Rechazo a un cliente que **no pagaba** (acierto) | Se evita una pérdida | `+ capital × perdida_no_pago` |
| Rechazo a un cliente que **sí pagaba** (falso positivo) | Se pierde el margen del crédito | `− capital × margen_buen_cliente` |
| Aprobación (cualquier caso) | Igual que la referencia | `0` |

- `valor_negocio`: valor neto en la moneda de la base (suma de ambos efectos).
- `pct_valor_capturado`: qué parte del valor máximo posible (rechazar exactamente a todos los morosos) logra el modelo.

> ⚠️ **Los dos porcentajes son supuestos**, porque la base no trae margen ni pérdida reales (la cuota por el plazo ni siquiera cubre el capital en el 91 % de los casos). Conviene validarlos con el área de negocio; el modelo y el umbral se recalculan solos al cambiarlos.

In [10]:
CONFIG_NEGOCIO = {
    "perdida_no_pago": 0.30,       # % del capital que se pierde si el cliente no paga a tiempo
    "margen_buen_cliente": 0.10,   # % del capital que se gana con un cliente que sí paga
}


def valor_negocio(y_true, y_pred, capital, cfg=CONFIG_NEGOCIO) -> dict:
    """Valor neto de usar el modelo frente a aprobar a todos.

    y_pred == 0 significa 'se rechaza' (el modelo predice que no paga).
    """
    y_true, y_pred, capital = map(np.asarray, (y_true, y_pred, capital))
    acierto = (y_pred == 0) & (y_true == 0)          # moroso rechazado
    falso_positivo = (y_pred == 0) & (y_true == 1)   # buen cliente rechazado
    ahorro = capital[acierto].sum() * cfg["perdida_no_pago"]
    costo = capital[falso_positivo].sum() * cfg["margen_buen_cliente"]
    maximo = capital[y_true == 0].sum() * cfg["perdida_no_pago"]  # modelo perfecto
    neto = ahorro - costo
    return {"valor_negocio": neto,
            "pct_valor_capturado": neto / maximo if maximo > 0 else 0.0}

## 3. Funciones reutilizables

- `score_no_pago`: puntaje continuo de "no paga" (para el ROC AUC y para el umbral).
- `summarize_classification`: métricas sobre la clase 0, más las de negocio si se entrega el capital. El ROC AUC usa el puntaje continuo; con clases duras subestima la capacidad de discriminar.

In [11]:
def score_no_pago(model, X) -> np.ndarray:
    """Puntaje continuo de 'no paga' (mayor = más riesgo), para el ROC AUC."""
    # Modelos con probabilidades: tomamos la columna de la clase 0
    if hasattr(model, "predict_proba"):
        clases = list(model.classes_)
        return model.predict_proba(X)[:, clases.index(0)]
    # LinearSVC no tiene probabilidades: una decisión alta significa clase 1
    # (paga), así que invertimos el signo para que "mayor = más riesgo"
    return -model.decision_function(X)


def summarize_classification(y_true, y_pred, y_score=None, capital=None,
                             cfg=CONFIG_NEGOCIO) -> dict:
    """Métricas sobre la clase 0 (no paga a tiempo) y, si hay capital, de negocio."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    # Si no hay puntaje continuo, el AUC se calcula con las clases duras
    roc_in = y_score if y_score is not None else (y_pred == 0).astype(int)
    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        # pos_label=0: medimos qué tan bien detectamos a quienes NO pagan
        "precision": precision_score(y_true, y_pred, pos_label=0, zero_division=0),
        "recall": recall_score(y_true, y_pred, pos_label=0, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, pos_label=0, zero_division=0),
        "roc_auc": roc_auc_score((y_true == 0).astype(int), roc_in),
        "casosNoPagoAtiempo": int(np.count_nonzero(y_pred == 0)),
    }
    if capital is not None:
        out.update(valor_negocio(y_true, y_pred, capital, cfg))
    return out

## 4. Modelos candidatos

Todos manejan el desbalance de clases (~5% no paga). En XGBoost, la clase positiva es la 1 (la mayoritaria), por eso `scale_pos_weight = n_neg / n_pos` la reduce.

In [12]:
def get_models(y_train) -> dict:
    """Modelos candidatos, todos con manejo del desbalance de clases."""
    n_neg, n_pos = (y_train == 0).sum(), (y_train == 1).sum()
    models = {
        # Los modelos lineales necesitan variables en la misma escala
        "logistic": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(solver="liblinear", class_weight="balanced")),
        ]),
        "svc": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LinearSVC(C=1.0, max_iter=5000, dual=False, class_weight="balanced")),
        ]),
        # Árbol y bosque: profundidad limitada para frenar el sobreajuste
        "decision_tree": DecisionTreeClassifier(
            max_depth=5, min_samples_leaf=10, class_weight="balanced",
            random_state=RANDOM_STATE),
        "random_forest": RandomForestClassifier(
            n_estimators=150, max_depth=7, min_samples_leaf=5, max_features="sqrt",
            class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
    }
    if XGBClassifier is not None:
        models["xgboost"] = XGBClassifier(
            eval_metric="logloss", n_estimators=200, max_depth=6, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=n_neg / n_pos,  # compensa el desbalance
            random_state=RANDOM_STATE)
    return models

## 5. Ajuste del umbral de decisión

El umbral por defecto (0,5) no tiene por qué ser el mejor: con clases balanceadas artificialmente, las probabilidades quedan infladas y se rechaza a demasiada gente. Se busca el umbral que **maximiza el valor de negocio**.

Para no hacer trampa, el umbral se elige con **predicciones fuera de muestra sobre train** (validación cruzada) y **test se usa una sola vez** para medir el resultado. Como los modelos no comparten escala de puntaje, los candidatos son cuantiles del propio puntaje (equivale a "rechazar al X % más riesgoso"); el último candidato es no rechazar a nadie.

In [13]:
def score_oof(model, X, y, cv) -> np.ndarray:
    """Puntaje de 'no paga' fuera de muestra (cada fila la predice un modelo que no la vio)."""
    metodo = "predict_proba" if hasattr(model, "predict_proba") else "decision_function"
    oof = cross_val_predict(clone(model), X, y, cv=cv, method=metodo)
    # predict_proba devuelve las clases ordenadas [0, 1]: la columna 0 es "no paga"
    return oof[:, 0] if metodo == "predict_proba" else -oof


def buscar_umbral(y_true, score, capital, cfg=CONFIG_NEGOCIO):
    """Devuelve (umbral, curva) con el umbral que maximiza el valor de negocio.

    Se rechaza (predice 'no paga') cuando score >= umbral. La curva guarda el
    valor, el % de rechazados y el recall de cada umbral candidato.
    """
    y_true, score, capital = map(np.asarray, (y_true, score, capital))
    candidatos = np.unique(np.quantile(score, np.linspace(0, 1, 401)))
    candidatos = np.append(candidatos, np.inf)  # último: no rechazar a nadie
    filas = []
    for t in candidatos:
        rechaza = score >= t
        pred = np.where(rechaza, 0, 1)
        filas.append({
            "umbral": t,
            "pct_rechazados": rechaza.mean(),
            "recall": rechaza[y_true == 0].mean(),
            "valor_negocio": valor_negocio(y_true, pred, capital, cfg)["valor_negocio"],
        })
    curva = pd.DataFrame(filas)
    # Ante un empate se elige el umbral más alto (el más conservador)
    mejor = np.where(curva["valor_negocio"] >= curva["valor_negocio"].max() - 1e-9)[0][-1]
    return float(curva.loc[mejor, "umbral"]), curva

## 6. `build_model`

Entrena un clasificador y devuelve:
- métricas en **train y test con el umbral ajustado**;
- métricas **fuera de muestra (`oof`)** sobre train, que sirven para elegir el modelo sin mirar test;
- las métricas de test con la **decisión por defecto** del modelo (`base_test`), para comparar contra el umbral ajustado.

In [14]:
def build_model(classifier, data: dict, cv_splits: int = 5,
                cfg: dict = CONFIG_NEGOCIO) -> dict:
    """Entrena un clasificador, ajusta su umbral y devuelve todas las métricas."""
    X_train, X_test = data["X_train"], data["X_test"]
    y_train, y_test = data["y_train"], data["y_test"]
    cap_train = X_train["capital_prestado"].to_numpy()
    cap_test = X_test["capital_prestado"].to_numpy()

    # 1) Umbral óptimo con predicciones fuera de muestra sobre train
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=RANDOM_STATE)
    s_oof = score_oof(classifier, X_train, y_train, cv)
    umbral, curva = buscar_umbral(y_train, s_oof, cap_train, cfg)

    # 2) Entrenamiento final con todo train
    classifier.fit(X_train, y_train)

    # 3) Métricas en train y test aplicando el umbral ajustado
    resumen = {}
    for nombre, X, y, cap in (("train", X_train, y_train, cap_train),
                              ("test", X_test, y_test, cap_test)):
        score = score_no_pago(classifier, X)
        pred = np.where(score >= umbral, 0, 1)
        resumen[nombre] = summarize_classification(y, pred, score, cap, cfg)

    # 4) Métricas fuera de muestra (criterio para elegir el modelo)
    resumen["oof"] = summarize_classification(
        y_train, np.where(s_oof >= umbral, 0, 1), s_oof, cap_train, cfg)

    # 5) Referencia: test con la decisión por defecto del modelo (sin ajustar)
    resumen["base_test"] = summarize_classification(
        y_test, classifier.predict(X_test), score_no_pago(classifier, X_test),
        cap_test, cfg)

    resumen["umbral"] = umbral
    resumen["curva"] = curva
    return resumen

## 7. Selección del mejor modelo y entrenamiento completo

**Criterio de selección:** mayor **valor de negocio fuera de muestra** (`oof`) sobre train. Así el modelo se elige sin mirar test, y test queda como estimación independiente del desempeño.

`ft_engineering()` excluye `puntaje` por defecto (separa perfectamente a pagadores y no pagadores, lo que apunta a fuga de información). Para incluirla: `train_all(excluir_features=[])`.

In [15]:
def select_best(resultados: dict) -> str:
    """Mejor modelo: mayor valor de negocio fuera de muestra (validación cruzada)."""
    return max(resultados, key=lambda m: resultados[m]["oof"]["valor_negocio"])


def train_all(cfg: dict = CONFIG_NEGOCIO, **kwargs) -> dict:
    """Entrena todos los modelos y devuelve lo que necesita model_evaluation.py.

    cfg: supuestos de negocio (ver CONFIG_NEGOCIO).
    kwargs se pasan a ft_engineering (p. ej. excluir_features=[]).
    """
    # 1) Datos limpios, divididos (estratificado) y transformados
    data = ft_engineering(**kwargs)

    # 2) Entrenar, ajustar el umbral y evaluar cada modelo
    models = get_models(data["y_train"])
    resultados, filas = {}, []
    for nombre, clf in models.items():
        resultados[nombre] = build_model(clf, data, cfg=cfg)
        # Formato largo (Model, Data Set, Metric, Score): es el que usan las
        # tablas y gráficos de model_evaluation.py
        for conjunto in ("train", "test"):
            for metrica, valor in resultados[nombre][conjunto].items():
                filas.append({"Model": nombre, "Data Set": conjunto,
                              "Metric": metrica, "Score": valor})
    results_df = pd.DataFrame(filas)

    # 3) Elegir el ganador
    best = select_best(resultados)

    return {
        "models": models,
        "results_df": results_df,
        "best": best,
        "data": data,
        "resultados": resultados,  # incluye oof, base_test, umbral y curva
        "umbrales": {m: r["umbral"] for m, r in resultados.items()},
        "config_negocio": cfg,
    }


def guardar(res: dict, carpeta: str = ".") -> None:
    """Guarda el mejor modelo, el preprocesador, el umbral y la tabla de resultados."""
    os.makedirs(carpeta, exist_ok=True)  # crea la carpeta si no existe
    best = res["best"]
    joblib.dump(res["models"][best], f"{carpeta}/modelo_{best}.joblib")
    joblib.dump(res["data"]["preprocessor"], f"{carpeta}/preprocessor.joblib")
    res["results_df"].to_csv(f"{carpeta}/resultados_modelos.csv", index=False)

    # Configuración para usar el modelo en producción: se rechaza (predice
    # "no paga") cuando score_no_pago >= umbral. Umbral None = no rechazar a nadie
    umbral = res["umbrales"][best]
    with open(f"{carpeta}/config_modelo.json", "w", encoding="utf-8") as f:
        json.dump({
            "modelo": best,
            "umbral": None if np.isinf(umbral) else umbral,
            "config_negocio": res["config_negocio"],
            "features": list(res["data"]["feature_names"]),
        }, f, ensure_ascii=False, indent=2)

## 8. Ejecución

Esta celda **solo corre al abrir el notebook directamente**. Cuando `model_evaluation.py` lo importa, el guard `__name__ == "__main__"` evita que se vuelva a entrenar todo.

In [16]:
if __name__ == "__main__":
    res = train_all()

    # Métricas de test con el umbral ajustado
    tabla = res["results_df"].pivot_table(
        index=["Model", "Data Set"], columns="Metric", values="Score")
    display(tabla[METRICAS + METRICAS_NEGOCIO].round(3))

    # Efecto del umbral: valor de negocio en test, decisión por defecto vs ajustado
    print("Valor de negocio en test (millones):")
    for m, r in res["resultados"].items():
        print(f"  {m:14s} por defecto: {r['base_test']['valor_negocio'] / 1e6:9.1f}"
              f" | umbral ajustado: {r['test']['valor_negocio'] / 1e6:7.1f}"
              f" | valor fuera de muestra (train): {r['oof']['valor_negocio'] / 1e6:7.1f}")

    print("\nMejor modelo:", res["best"])
    guardar(res, carpeta="../salidas")  # raíz del proyecto/salidas

Metric                  accuracy  precision  recall  f1_score  roc_auc  \
Model         Data Set                                                   
decision_tree test         0.952      0.400   0.039     0.071    0.612   
              train        0.954      0.604   0.078     0.139    0.738   
logistic      test         0.950      0.143   0.010     0.018    0.684   
              train        0.952      0.410   0.039     0.071    0.675   
random_forest test         0.947      0.276   0.078     0.122    0.691   
              train        0.965      0.734   0.399     0.517    0.932   
svc           test         0.953      0.000   0.000     0.000    0.684   
              train        0.952      0.000   0.000     0.000    0.674   
xgboost       test         0.941      0.179   0.069     0.099    0.651   
              train        0.991      0.863   0.968     0.912    0.999   

Metric                  casosNoPagoAtiempo  valor_negocio  pct_valor_capturado  
Model         Data Set                                                          
decision_tree test                    10.0   2.531611e+06                0.028  
              train                   53.0   2.957992e+07                0.088  
logistic      test                     7.0  -6.138088e+05               -0.007  
              train                   39.0   5.526607e+06                0.016  
random_forest test                    29.0   3.681710e+06                0.041  
              train                  222.0   1.474515e+08                0.438  
svc           test                     0.0   0.000000e+00                0.000  
              train                    0.0   0.000000e+00                0.000  
xgboost       test                    39.0   3.800436e+05                0.004  
              train                  459.0   3.067497e+08                0.912

Valor de negocio en test (millones):
  logistic       por defecto:    -143.7 | umbral ajustado:    -0.6 | valor fuera de muestra (train):     2.9
  svc            por defecto:    -144.9 | umbral ajustado:     0.0 | valor fuera de muestra (train):     0.0
  decision_tree  por defecto:    -125.9 | umbral ajustado:     2.5 | valor fuera de muestra (train):    13.2
  random_forest  por defecto:     -50.9 | umbral ajustado:     3.7 | valor fuera de muestra (train):    19.3
  xgboost        por defecto:     -12.3 | umbral ajustado:     0.4 | valor fuera de muestra (train):    13.8

Mejor modelo: random_forest
